In [21]:
import numpy as np
import yfinance as yf
import sympy as sp
import datetime as dt
import pandas as pd

In [50]:
def get_hist(ticker, start, end):
    hist = yf.Ticker(ticker).history(start=start,end=end, interval="3mo")
    new_index = hist.reset_index().Date.apply(lambda x: x.replace(tzinfo=None))
    
    if new_index[0] > start :
        first_trading_date = yf.Ticker(ticker).history(period="max", interval="1d").index[0]
        quarter_start_mo = (int((first_trading_date.month-1)/3+1)*3+1)%12
        new_start = dt.datetime(year=first_trading_date.year, month=quarter_start_mo, day=1)

        hist = yf.Ticker(ticker).history(start=new_start,end=end, interval="3mo")

    return hist

In [51]:
start= dt.datetime(year=2000, month=1, day=1)
end= dt.datetime(year=2021, month=1, day=1)

get_hist("AAPL", start, end)

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2000-01-01 00:00:00-05:00,0.785593,1.126422,0.647950,1.017338,28573316800,0.0000,0.0
2000-04-01 00:00:00-05:00,1.014996,1.044960,0.602069,0.784656,26575360000,0.0000,2.0
2000-07-01 00:00:00-04:00,0.780911,0.960688,0.380156,0.385773,25899787200,0.0000,0.0
2000-10-01 00:00:00-04:00,0.399819,0.400756,0.204123,0.222850,39253132800,0.0000,0.0
2001-01-01 00:00:00-05:00,0.222850,0.355811,0.216296,0.330642,31532289600,0.0000,0.0
...,...,...,...,...,...,...,...
2019-10-01 00:00:00-04:00,54.091575,70.650463,51.702672,70.573555,6615331600,0.1925,0.0
2020-01-01 00:00:00-05:00,71.409784,79.029500,51.250455,61.297577,12233722000,0.1925,0.0
2020-04-01 00:00:00-04:00,59.560790,89.976663,57.241180,88.145134,9314610800,0.2050,0.0


In [ ]:
def returns(hist):
    array = [(row.Close - row.Open)/row.Open for row in hist.iloc]
    return array

def expected_returns(hist):
    array = returns(hist)
    return np.mean(array)

def stock_covar(hist1, hist2):
    common = hist1.merge(hist2, 'inner', left_on=hist1.index, right_on=hist2.index)
    common_dates = common.key_0
    
    hist1 = hist1.loc[hist1.index >= common_dates[0]]
    hist2 = hist2.loc[hist2.index >= common_dates[0]]
    
    array1 = returns(hist1)
    array2 = returns(hist2)

    return np.cov(array1, array2)

In [67]:
expected_returns(get_hist("GOOG", start, end))

np.float64(0.06273555460177029)

In [73]:
hist1 = get_hist("GOOG", start, end)
hist2 = get_hist("AAPL", start, end)

stock_covar(hist1, hist2)

array([[0.02752362, 0.01527851],
       [0.01527851, 0.03871145]])

In [96]:
all_stocks = ['AAPL', "GOOG", "NVDA", "CL", "BA", "NKE", "DB"]

def generate_Sigma(all_stocks, start, end):
    n = len(all_stocks)
    covar = np.zeros((n, n))

    combs = []

    for i, t1 in enumerate(all_stocks):
        for j, t2 in enumerate(all_stocks):
            if ((t1, t2) in combs) or ((t2, t1) in combs) or t1 == t2:
                continue
            combs.append((t1, t2))
            c = stock_covar(get_hist(t1, start, end), get_hist(t2, start, end))

            covar[i, i] = c[0,0]
            covar[i, j] = c[0, 1]
            covar[j, i] = c[1, 0]
            covar[j, j] = c[1, 1]

    return covar

In [97]:
w = np.array([expected_returns(get_hist(stock, start, end)) for stock in all_stocks])

covar = generate_Sigma(all_stocks, start, end)

In [142]:
def benchmark(stocks, weights, benchmark_ticker, start, end):
    dates_array = yf.Ticker(stocks[0]).history(start=start,end=end, interval="1d").index

    df = pd.DataFrame(0, index=dates_array, columns=["portfolio", "benchmark"])
    
    for i, ticker in enumerate(stocks):
        hist = yf.Ticker(ticker).history(start=start,end=end, interval="1d")

        df.portfolio += weights[i]*hist.Open/hist.iloc[0].Open*100

    df.benchmark = yf.Ticker(benchmark_ticker).history(start=start, end=end, interval="1d").Open
    df.benchmark /= df.iloc[0].benchmark
    df.benchmark *= 100

    return df


In [ ]:
#weights = np.ones(len(all_stocks))/len(all_stocks)

df = benchmark(all_stocks, weights, "SPY", end, dt.datetime.today())


In [149]:
import plotly.express as px

px.line(df)

In [ ]:
df.benchmark /= df.iloc[0].benchmark

df.benchmark

Date
2021-01-04 00:00:00-05:00    0.010000
2021-01-05 00:00:00-05:00    0.009808
2021-01-06 00:00:00-05:00    0.009851
2021-01-07 00:00:00-05:00    0.010021
2021-01-08 00:00:00-05:00    0.010141
                               ...   
2026-03-17 00:00:00-04:00    0.019192
2026-03-18 00:00:00-04:00    0.019077
2026-03-19 00:00:00-04:00    0.018752
2026-03-20 00:00:00-04:00    0.018790
2026-03-23 00:00:00-04:00    0.018835
Name: benchmark, Length: 1310, dtype: float64